# RideFlow — 02 Bronze (marketplace)

Typed, validated Bronze from LANDING.

Runs the LakeLogic `snowflake` engine (schema casts, row rules, quarantine, transforms, gold SQL, SCD2) for this layer on the current session, and persists each table.


In [ ]:
# Trial-safe install: External Access Integrations aren't allowed on trial accounts,
# so notebooks can't pip-install from the internet. Instead we install LakeLogic +
# pure-python deps from wheels staged in Snowflake (@PIPELINE.WHEELS). The container
# runtime already ships pandas / pydantic / pyyaml / snowflake-connector.
import os, subprocess, sys
from snowflake.snowpark.context import get_active_session
_session = get_active_session()
os.makedirs('/tmp/wl', exist_ok=True)
_session.file.get('@RIDEFLOW_DEV_DEMO.PIPELINE.WHEELS', '/tmp/wl')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-index',
                '--find-links', '/tmp/wl', '--no-deps', 'lakelogic', 'loguru', 'sqlglot'], check=True)


In [ ]:
import os, sys
# Resolve the repo root (folder containing domains_rideflow) from the Git-connected
# Workspace checkout, so runtime.* imports and relative contract paths both resolve.
d = os.getcwd()
while d != os.path.dirname(d) and not os.path.isdir(os.path.join(d, 'domains_rideflow')):
    d = os.path.dirname(d)
REPO = d; sys.path.insert(0, REPO); os.chdir(REPO)

DB  = 'RIDEFLOW_DEV_DEMO'
SYS = 'domains_rideflow/marketplace/rideflow/_system.yaml'
os.environ['SNOWFLAKE_DATABASE'] = DB   # contracts resolve {catalog} from this

from snowflake.snowpark.context import get_active_session
session = get_active_session()
conn = session.connection            # reuse THIS session (no password in the notebook)
from lakelogic.engines.snowflake import SnowflakeAdapter
SnowflakeAdapter.set_shared_connection(conn)
print('repo:', REPO)


In [ ]:
from runtime.medallion import run_layer
results = run_layer(conn, DB, SYS, 'bronze')
import pandas as pd
pd.DataFrame(results, columns=['entity','status','good','quarantined'])
